## Google Colab Setup

This notebook can run either locally from the repo or on Google Colab.

Colab workflow:
- Mount Google Drive and save all trained-model outputs there immediately.
- Download only the exact code, data, RSA, and checkpoint files needed into `/content/XAllergen2.0`.
- Download a fresh ESM-2 snapshot from Hugging Face.
- Avoid `git clone`, which can be less reliable in hosted notebook runtimes.

Required GitHub-hosted artifacts:
- `models/baseline_frozen_esm2.pt`
- `data/rsa/deepalgpro_train_rsa.json.gz`
- `data/rsa/deepalgpro_test_rsa.json.gz`


In [1]:
import os
import shutil
import sys
import urllib.request
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
os.environ['XALLERGEN_RUN_TARGET'] = 'colab' if IS_COLAB else 'local'

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/XAllergen2.0')
else:
    DRIVE_ROOT = None

if IS_COLAB:
    RUNTIME_ROOT = Path('/content/XAllergen2.0')
else:
    for _candidate in [Path.cwd(), *Path.cwd().parents]:
        if (_candidate / 'src' / 'xallergen').exists():
            RUNTIME_ROOT = _candidate
            break
    else:
        raise FileNotFoundError('Could not locate repo root with src/xallergen')

SRC_DIR = RUNTIME_ROOT / 'src'
PACKAGE_DIR = SRC_DIR / 'xallergen'
DATA_DIR = RUNTIME_ROOT / 'data'
RSA_DIR = DATA_DIR / 'rsa'
RUNTIME_MODELS_DIR = RUNTIME_ROOT / 'models'
RUNTIME_RESULTS_DIR = RUNTIME_ROOT / 'results'
HF_DIR = RUNTIME_ROOT / 'hf_models' / 'facebook_esm2_t6_8M_UR50D'

if IS_COLAB:
    MODELS_DIR = DRIVE_ROOT / 'models'
    RESULTS_DIR = DRIVE_ROOT / 'results'
else:
    MODELS_DIR = RUNTIME_MODELS_DIR
    RESULTS_DIR = RUNTIME_RESULTS_DIR

for path in [SRC_DIR, PACKAGE_DIR, DATA_DIR, RSA_DIR, RUNTIME_MODELS_DIR, RUNTIME_RESULTS_DIR, MODELS_DIR, RESULTS_DIR, HF_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if IS_COLAB:
    from huggingface_hub import snapshot_download

    RAW = 'https://raw.githubusercontent.com/Jeffateth/XAllergen2.0/main'

    package_files = [
        '__init__.py',
        'baseline_notebook_utils.py',
        'baseline_sasa_experiment.py',
        'mtl_epitope_notebook_utils.py',
    ]
    data_files = [
        'deepalgpro_train_cleaned.csv',
        'deepalgpro_test_cleaned.csv',
        'positives_splitB.csv',
    ]
    rsa_files = [
        'deepalgpro_train_rsa.json.gz',
        'deepalgpro_test_rsa.json.gz',
    ]

    download_jobs = []
    download_jobs.extend((f'{RAW}/src/xallergen/{name}', PACKAGE_DIR / name) for name in package_files)
    download_jobs.extend((f'{RAW}/data/{name}', DATA_DIR / name) for name in data_files)
    download_jobs.extend((f'{RAW}/data/rsa/{name}', RSA_DIR / name) for name in rsa_files)
    download_jobs.append((f'{RAW}/models/baseline_frozen_esm2.pt', RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'))

    failed_urls = []
    for url, dst in download_jobs:
        try:
            urllib.request.urlretrieve(url, dst)
        except Exception as exc:
            failed_urls.append((url, str(exc)))

    if failed_urls:
        details = '\n'.join(f'  - {url} -> {err}' for url, err in failed_urls)
        raise RuntimeError('Failed to download required GitHub artifacts:\n' + details)

    fresh_model_path = snapshot_download(
        repo_id='facebook/esm2_t6_8M_UR50D',
        local_dir=HF_DIR,
        local_dir_use_symlinks=False,
        force_download=True,
        resume_download=False,
    )
    os.environ['XALLERGEN_HF_MODEL_DIR'] = str(fresh_model_path)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'RUN_TARGET: {os.environ["XALLERGEN_RUN_TARGET"]}')
print(f'Runtime root: {RUNTIME_ROOT}')
print(f'Data dir: {DATA_DIR}')
print(f'RSA dir: {RSA_DIR}')
print(f'Runtime models dir: {RUNTIME_MODELS_DIR}')
print(f'Runtime results dir: {RUNTIME_RESULTS_DIR}')
print(f'Output models dir: {MODELS_DIR}')
print(f'Output results dir: {RESULTS_DIR}')
print(f'Source package dir: {PACKAGE_DIR}')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

RUN_TARGET: colab
Runtime root: /content/XAllergen2.0
Data dir: /content/XAllergen2.0/data
RSA dir: /content/XAllergen2.0/data/rsa
Models dir: /content/XAllergen2.0/models
Source package dir: /content/XAllergen2.0/src/xallergen


# Baseline ESM-2 with RSA Regularization

This notebook extends the frozen ESM-2 baseline with a single RSA regularization sweep driven by precomputed NetSurfP RSA values.

Design notes:
- The original `03_baseline_model_esm2.ipynb` remains the baseline reference.
- `lambda_rsa=0.0` is the classification-only baseline within the same sweep.
- The model stays unchanged: frozen ESM-2 backbone, attention pooling, and the existing classification head only.
- The RSA regularizer uses fixed precomputed RSA values from the split `.json.gz` artifacts and constrains attention pooling weights through the loss term.


In [2]:
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split

from xallergen.baseline_notebook_utils import (
    RANDOM_STATE,
    build_tokenizer,
    load_baseline_checkpoint,
    seed_everything,
)
from xallergen.baseline_sasa_experiment import (
    RSARegularizationConfig,
    build_dataloader,
    compute_metrics,
    inspect_rsa_inputs,
    load_rsa_lookup_dicts,
    predict,
    run_lambda_rsa_sweep,
)

TRAIN_CSV = DATA_DIR / 'deepalgpro_train_cleaned.csv'
TEST_CSV = DATA_DIR / 'deepalgpro_test_cleaned.csv'
TRAIN_RSA_PATH = RSA_DIR / 'deepalgpro_train_rsa.json.gz'
TEST_RSA_PATH = RSA_DIR / 'deepalgpro_test_rsa.json.gz'
POSITIVES_SPLITB_CSV = DATA_DIR / 'positives_splitB.csv'
BASELINE_CHECKPOINT_PATH = RUNTIME_MODELS_DIR / 'baseline_frozen_esm2.pt'
RSA_RESULTS_DIR = RESULTS_DIR / 'rsa_regularization'
RSA_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_SUMMARY_PATH = RSA_RESULTS_DIR / 'sweep_summary.csv'

for required_path in [
    TRAIN_CSV,
    TEST_CSV,
    TRAIN_RSA_PATH,
    TEST_RSA_PATH,
    POSITIVES_SPLITB_CSV,
    BASELINE_CHECKPOINT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(f'Missing required file: {required_path}')

CONFIG = RSARegularizationConfig(
    lambda_cls=1.0,
    lambda_rsa_values=(0.0, 0.1, 0.5, 1.0, 5.0),
    add_special_tokens=False,
)

seed_everything(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'Tokenizer adds special tokens: {CONFIG.add_special_tokens}')
print(f'Baseline checkpoint: {BASELINE_CHECKPOINT_PATH}')
print(f'Sweep checkpoints + metrics dir: {RSA_RESULTS_DIR}')
if IS_COLAB:
    print(f'Google Drive root: {DRIVE_ROOT}')


Device: cuda
Tokenizer adds special tokens: False
Baseline checkpoint: /content/XAllergen2.0/models/baseline_frozen_esm2.pt
RSA results dir: /content/XAllergen2.0/results/rsa_regularization


In [3]:
train_df = pd.read_csv(TRAIN_CSV).copy()
test_df = pd.read_csv(TEST_CSV).copy()
for df in [train_df, test_df]:
    df['sequence_id'] = df['sequence_id'].astype(str).str.strip()
    df['sequence'] = df['sequence'].astype(str).str.strip().str.upper()
    df['label'] = df['label'].astype(int)

rsa_report_df = inspect_rsa_inputs(
    train_rsa_path=TRAIN_RSA_PATH,
    test_rsa_path=TEST_RSA_PATH,
    train_frame=train_df,
    test_frame=test_df,
)
display(rsa_report_df)

print(f'Train RSA file: {TRAIN_RSA_PATH}')
print(f'Test RSA file : {TEST_RSA_PATH}')


,path,format,compressed,n_sequences,rsa_min,rsa_max,rsa_in_unit_interval,min_length,max_length,expected_sequences,missing_sequences,extra_sequences,exact_id_match
0,/content/XAllergen2.0/data/rsa/deepalgpro_trai...,precomputed_rsa_json,True,5551,0.000877,0.978356,True,11,999,5551,0,0,True
1,/content/XAllergen2.0/data/rsa/deepalgpro_test...,precomputed_rsa_json,True,1377,0.000882,0.978507,True,11,979,1377,0,0,True


Train RSA file: /content/XAllergen2.0/data/rsa/deepalgpro_train_rsa.json.gz
Test RSA file : /content/XAllergen2.0/data/rsa/deepalgpro_test_rsa.json.gz


In [4]:
train_split_df, val_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df['label'],
)
train_split_df = train_split_df.reset_index(drop=True).copy()
val_split_df = val_split_df.reset_index(drop=True).copy()

print(f'Train sequences: {len(train_split_df):,}')
print(f'Validation sequences: {len(val_split_df):,}')
print(f'Held-out test sequences: {len(test_df):,}')


Train sequences: 4,995
Validation sequences: 556
Held-out test sequences: 1,377


In [5]:
train_rsa_lookup, test_rsa_lookup, rsa_summary = load_rsa_lookup_dicts(
    train_rsa_path=TRAIN_RSA_PATH,
    test_rsa_path=TEST_RSA_PATH,
    train_frame=train_df,
    test_frame=test_df,
    add_special_tokens=CONFIG.add_special_tokens,
)
display(pd.DataFrame([rsa_summary['train'], rsa_summary['test']]))


,path,format,compressed,n_expected_sequences,n_parsed_sequences,n_missing_sequences,n_extra_sequences,n_length_mismatches,missing_sequence_ids,extra_sequence_ids,length_mismatches,add_special_tokens
0,/content/XAllergen2.0/data/rsa/deepalgpro_trai...,precomputed_rsa_json,True,5551,5551,0,0,0,[],[],[],False
1,/content/XAllergen2.0/data/rsa/deepalgpro_test...,precomputed_rsa_json,True,1377,1377,0,0,0,[],[],[],False


In [6]:
tokenizer = build_tokenizer()
val_loader_reference = build_dataloader(
    frame=val_split_df,
    rsa_lookup=train_rsa_lookup,
    tokenizer=tokenizer,
    batch_size=CONFIG.batch_size,
    shuffle=False,
    add_special_tokens=CONFIG.add_special_tokens,
)

baseline_model, _ = load_baseline_checkpoint(BASELINE_CHECKPOINT_PATH, DEVICE)
baseline_val_summary, baseline_val_predictions_df = predict(
    baseline_model,
    val_loader_reference,
    DEVICE,
    criterion=None,
    lambda_cls=CONFIG.lambda_cls,
    lambda_rsa=0.0,
    threshold=CONFIG.threshold,
)
baseline_val_metrics = compute_metrics(baseline_val_predictions_df, threshold=CONFIG.threshold)
print('Reference baseline checkpoint validation AUROC:', round(baseline_val_metrics['auroc'], 3))


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Reference baseline checkpoint validation AUROC: 0.978


In [7]:
rsa_summary_df = run_lambda_rsa_sweep(
    config=CONFIG,
    train_split_df=train_split_df,
    val_split_df=val_split_df,
    test_df=test_df,
    train_rsa_lookup=train_rsa_lookup,
    test_rsa_lookup=test_rsa_lookup,
    positives_splitb_csv=POSITIVES_SPLITB_CSV,
    output_dir=RSA_RESULTS_DIR,
    device=DEVICE,
)
sweep_summary_df = rsa_summary_df.loc[:, [
    'lambda_rsa',
    'epoch',
    'val_auroc',
    'val_precision',
    'val_recall',
    'val_f1',
    'val_mcc',
    'val_accuracy',
    'test_auroc',
    'test_precision',
    'test_recall',
    'test_f1',
    'test_mcc',
    'test_accuracy',
    'residue_auroc',
    'residue_auprc',
    'residue_precision_at_k',
]].copy()
SWEEP_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)
sweep_summary_df.to_csv(SWEEP_SUMMARY_PATH, index=False)
display(rsa_summary_df)
display(sweep_summary_df)


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


lambda_rsa=0:   0%|          | 0/30 [00:00<?, ?epoch/s]

Epoch   1/30 | train_cls=0.43256 | train_rsa=None | train_total=0.43256 | val_cls=0.33752 | val_rsa=None | val_total=0.33752 | val_auroc=0.94359 | best=1
Epoch   2/30 | train_cls=0.31299 | train_rsa=None | train_total=0.31299 | val_cls=0.27909 | val_rsa=None | val_total=0.27909 | val_auroc=0.95712 | best=2
Epoch   3/30 | train_cls=0.27837 | train_rsa=None | train_total=0.27837 | val_cls=0.27041 | val_rsa=None | val_total=0.27041 | val_auroc=0.96114 | best=3
Epoch   4/30 | train_cls=0.25671 | train_rsa=None | train_total=0.25671 | val_cls=0.25006 | val_rsa=None | val_total=0.25006 | val_auroc=0.96522 | best=4
Epoch   5/30 | train_cls=0.23961 | train_rsa=None | train_total=0.23961 | val_cls=0.25322 | val_rsa=None | val_total=0.25322 | val_auroc=0.96283 | best=4
Epoch   6/30 | train_cls=0.22956 | train_rsa=None | train_total=0.22956 | val_cls=0.23530 | val_rsa=None | val_total=0.23530 | val_auroc=0.96843 | best=6
Epoch   7/30 | train_cls=0.21919 | train_rsa=None | train_total=0.21919 | va

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

...

EsmModel LOAD REPORT from: /content/XAllergen2.0/hf_models/facebook_esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Evaluating proteins:   0%|          | 0/61 [00:00<?, ?it/s]

Evaluating proteins: processed 5/61 proteins
Evaluating proteins: processed 10/61 proteins
Evaluating proteins: processed 15/61 proteins
Evaluating proteins: processed 20/61 proteins
Evaluating proteins: processed 25/61 proteins
Evaluating proteins: processed 30/61 proteins
Evaluating proteins: processed 35/61 proteins
Evaluating proteins: processed 40/61 proteins
Evaluating proteins: processed 45/61 proteins
Evaluating proteins: processed 50/61 proteins
Evaluating proteins: processed 55/61 proteins
Evaluating proteins: processed 60/61 proteins
Evaluating proteins: processed 61/61 proteins


,lambda_rsa,epoch,val_auroc,val_f1,test_auroc,test_f1,test_mcc,residue_auroc,residue_auprc,residue_precision_at_k,checkpoint_path,metrics_path,probe_rows_path
0,0.0,16,0.977983,0.921676,0.969860,0.915997,0.833569,0.4677,0.2529,0.2163,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...
1,0.1,16,0.977815,0.923358,0.969347,0.913869,0.829261,0.4679,0.2527,0.2165,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...
2,0.5,21,0.978617,0.928177,0.969732,0.921817,0.847402,0.4660,0.2519,0.2148,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...
3,1.0,16,0.977012,0.925046,0.970542,0.918442,0.839039,0.4667,0.2517,0.2188,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...
4,5.0,21,0.977517,0.928177,0.969447,0.921248,0.845968,0.4570,0.2474,0.2146,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...,/content/XAllergen2.0/results/rsa_regularizati...


,lambda_rsa,val_auroc,val_f1,epoch
0,0.0,0.977983,0.921676,16
1,0.1,0.977815,0.923358,16
2,0.5,0.978617,0.928177,21
3,1.0,0.977012,0.925046,16
4,5.0,0.977517,0.928177,21


In [ ]:
lambda_zero_row = sweep_summary_df.loc[sweep_summary_df['lambda_rsa'].eq(0.0)].copy()
if not lambda_zero_row.empty:
    sweep_val_auroc = float(lambda_zero_row.iloc[0]['val_auroc'])
    print('Reference baseline checkpoint val AUROC:', round(baseline_val_metrics['auroc'], 3))
    print('lambda_rsa=0 sweep val AUROC        :', round(sweep_val_auroc, 3))
    print('Match to three decimals             :', round(baseline_val_metrics['auroc'], 3) == round(sweep_val_auroc, 3))
print(f'Saved sweep summary to: {SWEEP_SUMMARY_PATH}')
print(f'Sweep directory: {RSA_RESULTS_DIR}')
print('Best checkpoints are saved during training as each lambda improves validation loss.')
if IS_COLAB:
    print(f'All outputs were written directly to Google Drive under: {DRIVE_ROOT}')


Reference baseline checkpoint val AUROC: 0.978
lambda_rsa=0 sweep val AUROC        : 0.978
Match to three decimals             : True
Saved sweep summary to: /content/XAllergen2.0/results/rsa_regularization/sweep_summary.csv


: 

In [ ]:
# Shut down the runtime after everything above has finished.
import os
os.kill(os.getpid(), 9)

: 

: 